In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import viser
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
import viser.transforms as vtf
import open3d as o3d
import json

from nerfstudio.data.scene_box import SceneBox
from nerfstudio.cameras.cameras import Cameras, CameraType
from nerfstudio.utils.spherical_harmonics import SH2RGB

from shadow_splat.model import ShadowSplatModel, ShadowSplatModelConfig
from shadow_splat.util.covariance_utils import compute_cov

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Setup scene and model


In [ ]:
config = ShadowSplatModelConfig()
scene_box = SceneBox(aabb=torch.tensor([[-1, -1, -1], [1, 1, 1]]))
model = ShadowSplatModel(config, scene_box, num_train_data=100).to(device)

In [ ]:
# transforms = json.load(open("../data/master_chief/transforms.json"))
transforms = json.load(open("../data/master_chief_eevee/transforms.json"))

In [ ]:
pcd = o3d.io.read_point_cloud("../data/master_chief/all_points.ply")
points = torch.from_numpy(np.asarray(pcd.points)).float()
colors = 255 * torch.from_numpy(np.asarray(pcd.colors)).float()

model.seed_points = (points, colors)
model.populate_modules()
model.training = False
model.gauss_params["scales"] = torch.log(1e-3 * torch.ones(len(points), 3))
model.gauss_params["opacities"] = torch.logit(0.9 * torch.ones(len(points), 1))
model = model.to(device)

## Camera


In [ ]:
i = 70

c2w = transforms["frames"][i]["transform_matrix"]
c2w = torch.tensor(c2w).float().unsqueeze(0)

camera = Cameras(
    camera_to_worlds=c2w,
    fx=transforms["fl_x"],
    fy=transforms["fl_y"],
    cx=transforms["cx"],
    cy=transforms["cy"],
    width=int(transforms["w"]),
    height=int(transforms["h"]),
)
camera.to(device)

In [ ]:
outputs = model.get_outputs_for_camera(camera)

In [ ]:
plt.imshow(outputs["rgb"].detach().cpu())
plt.imsave(f"render_{i:03d}.png", outputs["rgb"].detach().cpu())

In [ ]:
# Create figure with two subplots side by side
plt.figure(figsize=(12, 6))

# Plot rendered image
plt.subplot(121)
plt.imshow(outputs["rgb"].detach().cpu())
plt.title("Rendered Image")
plt.axis("off")

# Plot ground truth image
plt.subplot(122)
# gt_img = plt.imread(f"../data/master_chief/images/view_{i:03d}.png")
gt_img = plt.imread(f"../data/master_chief_eevee/images/view_{i:03d}.png")
plt.imshow(gt_img)
plt.title("Ground Truth Image")
plt.axis("off")

plt.tight_layout()
plt.savefig(f"comparison_{i:03d}.png")

In [ ]:
# Load and blend images using PIL
from PIL import Image

# Open images and convert to RGBA
img1 = Image.open(f"render_{i:03d}.png").convert("RGBA")
img2 = Image.open(f"../data/master_chief/images/view_{i:03d}.png").convert("RGBA")

# Resize second image to match first if needed
img2 = img2.resize(img1.size)

# Create blended overlay
# blended = Image.alpha_composite(img1, img2)
blended = Image.blend(img1, img2, alpha=0.5)

# Save and display result
blended.save(f"overlay_{i:03d}.png")
plt.figure(figsize=(6, 6))
plt.imshow(blended)
plt.title("Overlaid Images (Rendered + Original)")
plt.axis("off")
plt.tight_layout()


In [ ]:
# Facing x-axis with z=0.5
# R = torch.tensor([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
# t = torch.tensor([[1.0], [0.0], [0.5]])
R = torch.tensor(
    [
        [-2.1618e-07, -1.6248e-02, 9.9987e-01],
        [1.0000e00, -3.5125e-09, 2.1615e-07],
        [-5.5511e-17, 9.9987e-01, 1.6248e-02],
    ]
)
t = torch.tensor([[5.1955e00], [1.1232e-06], [8.4427e-02]])

# Combine into camera_to_worlds matrix
camera_to_worlds = torch.cat([R, t], dim=1)
camera_to_worlds = camera_to_worlds.unsqueeze(0)  # Add batch dimension

# Define camera intrinsics
H = 360  # Height
W = 640  # Width
FOV = 60  # Field of view in degrees

# Convert FOV to radians and compute focal length
focal_length = W / (2 * np.tan(np.deg2rad(FOV / 2)))
fx = focal_length
fy = focal_length
cx = W / 2  # Principal point at image center
cy = H / 2

print(f"fx: {fx}, fy: {fy}, cx: {cx}, cy: {cy}")

# camera = Cameras(
#     camera_to_worlds=camera_to_worlds,
#     fx=fx,
#     fy=fy,
#     cx=cx,
#     cy=cy,
#     width=W,
#     height=H,
# ).to(device)
camera = Cameras(
    camera_to_worlds=camera_to_worlds,
    fx=1334.5028,
    fy=1334.5028,
    cx=637.0,
    cy=1024.0,
    width=1274,
    height=2048,
).to(device)

## Render


In [ ]:
outputs = model.get_outputs_for_camera(camera)

In [ ]:
outputs["rgb"].shape

In [ ]:
plt.imshow(outputs["rgb"].detach().cpu())

In [ ]:
viser_server = viser.ViserServer(port=7007)
viser_server.gui.configure_theme(dark_mode=True)

In [ ]:
centers = model.means.detach().clone().cpu().numpy()
colors = SH2RGB(model.features_dc.detach().clone().cpu().numpy())
opacities = torch.sigmoid(model.opacities.detach().clone()).cpu().numpy()
scales = torch.exp(model.scales.detach().clone())
covs = compute_cov(model.quats.detach().clone(), scales).cpu().numpy()

In [ ]:
splat_handle = viser_server.scene.add_gaussian_splats(
    "splats",
    centers=centers,
    rgbs=colors,
    opacities=opacities,
    covariances=covs,
)

In [ ]:
c2w = camera.camera_to_worlds.cpu().numpy()[0]
R = vtf.SO3.from_matrix(c2w[:3, :3])
R = R @ vtf.SO3.from_x_radians(np.pi)

viser_server.scene.add_camera_frustum(
    name="camera",
    fov=np.deg2rad(FOV),
    aspect=W / H,
    scale=0.1,
    color=(0.2, 0.2, 1.0),
    wxyz=R.wxyz,
    position=c2w[:3, 3],
)

## Light source


In [ ]:
R = torch.tensor([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
t = torch.tensor([[2.0], [0.0], [0.5]])

# Combine into camera_to_worlds matrix
camera_to_worlds = torch.cat([R, t], dim=1)
camera_to_worlds = camera_to_worlds.unsqueeze(0)  # Add batch dimension

# Define camera intrinsics
H = 360  # Height
W = 640  # Width
FOV = 60  # Field of view in degrees

# Convert FOV to radians and compute focal length
focal_length = W / (2 * np.tan(np.deg2rad(FOV / 2)))
fx = focal_length
fy = focal_length
cx = W / 2  # Principal point at image center
cy = H / 2

light_source = Cameras(
    camera_to_worlds=camera_to_worlds,
    fx=fx,
    fy=fy,
    cx=cx,
    cy=cy,
    width=W,
    height=H,
    camera_type=CameraType.PERSPECTIVE,
).to(device)

In [ ]:
model.update_light_source(light_source)

In [ ]:
outputs = model(camera)
plt.imshow(outputs["rgb"].detach().cpu())

In [ ]:
camera

# Nerfview


In [ ]:
from typing import Tuple
import numpy as np
import torch
import viser
import nerfview
from nerfstudio.cameras.cameras import Cameras, CameraType


def render_fn(
    camera_state: nerfview.CameraState, render_tab_state: nerfview.RenderTabState
) -> np.ndarray:
    # Parse camera state for camera-to-world matrix (c2w) and intrinsic (K) as
    # float64 numpy arrays.
    if render_tab_state.preview_render:
        width = render_tab_state.render_width
        height = render_tab_state.render_height
    else:
        width = render_tab_state.viewer_width
        height = render_tab_state.viewer_height

    c2w = camera_state.c2w
    R = vtf.SO3.from_matrix(c2w[:3, :3])
    R = R @ vtf.SO3.from_x_radians(np.pi)
    c2w[:3, :3] = R.as_matrix()

    K = camera_state.get_K([width, height])

    # Convert camera parameters to Cameras object
    camera_to_worlds = torch.from_numpy(c2w).unsqueeze(0)
    fx = K[0, 0]
    fy = K[1, 1]
    cx = K[0, 2]
    cy = K[1, 2]

    cameras = Cameras(
        camera_to_worlds=camera_to_worlds,
        fx=fx,
        fy=fy,
        cx=cx,
        cy=cy,
        width=width,
        height=height,
        camera_type=CameraType.PERSPECTIVE,
    ).to(device)

    print(cameras)

    # Render using the model
    outputs = model.get_outputs_for_camera(cameras)
    img = outputs["rgb"].cpu().numpy()[0]

    # Convert to uint8
    img = (img * 255).astype(np.uint8)
    return img


viser_server.scene.world_axes.visible = True
viewer = nerfview.Viewer(server=viser_server, render_fn=render_fn, mode="rendering")